# 📥 1_Extraccion — ETL Banano Ecuador

**Tarea del Job:** `tarea_extraccion`

Descarga los archivos de las 4 fuentes públicas del sector bananero ecuatoriano
(ESPAC, SIPA, FAOSTAT, AEBE) usando un agente personalizado en Python puro
(sin frameworks de orquestación) con Llama 4 Maverick para decidir qué archivos
son relevantes. Controla duplicados por hash MD5.

**Salidas:**
- Archivos nuevos en `/Volumes/workspace/default/raw_*`
- Tabla de control `bd_banano_ec.control_descargas_fuentes`
- Task Values: `hay_nuevos`, `total_archivos` (usados por la condición del DAG)


## Instalación de librerías
⚠️ Ejecutar solo esta celda primero, esperar el reinicio del kernel.

In [ ]:
%pip install langchain langchain-databricks beautifulsoup4 openpyxl xlrd
dbutils.library.restartPython()


## Configuración e inicialización
LLM (Llama 4 Maverick), token FAOSTAT, rutas de volúmenes y base de datos.

In [ ]:
import os, re, json, time, hashlib
import requests
from datetime import datetime
from typing import TypedDict, Optional, List
from bs4 import BeautifulSoup

# LangChain + Databricks Foundation Models
from langchain_databricks import ChatDatabricks
from langchain_core.messages import HumanMessage

# PySpark
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, IntegerType
)

# ── Rutas de volúmenes Databricks ─────────────────────────────────────────
RAW_PATH_ESPAC   = "/Volumes/workspace/default/raw_espac"
RAW_PATH_SIPA    = "/Volumes/workspace/default/raw_sipa"
RAW_PATH_FAOSTAT = "/Volumes/workspace/default/raw_faostat"
RAW_PATH_AEBE    = "/Volumes/workspace/default/raw_aebe"

# ── Configuración de tablas ────────────────────────────────────────────────
DB_NAME       = "bd_banano_ec"
CONTROL_TABLE = f"{DB_NAME}.control_descargas_fuentes"

HEADERS_HTTP = {"User-Agent": "Mozilla/5.0 (compatible; ETL-Banano/1.0)"}

# ── Credenciales FAOSTAT (auto-renovación de token) ───────────────────────
FAOSTAT_USERNAME = "secreto@gmail.com"
FAOSTAT_PASSWORD = "secreto"  # ⚠️ CAMBIAR POR TU PASSWORD REAL
FAOSTAT_TOKEN = None
FAOSTAT_TOKEN_EXPIRY = None

def get_faostat_token():
    """
    Obtiene o renueva el token de FAOSTAT automáticamente.
    El token dura 60 minutos - esta función lo renueva cuando expira.
    """
    global FAOSTAT_TOKEN, FAOSTAT_TOKEN_EXPIRY

    if FAOSTAT_TOKEN and FAOSTAT_TOKEN_EXPIRY:
        if time.time() < (FAOSTAT_TOKEN_EXPIRY - 300):
            return FAOSTAT_TOKEN

    print("  🔄 Renovando token FAOSTAT...")
    try:
        resp = requests.post(
            "https://faostatservices.fao.org/api/v1/auth/login",
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            data={"username": FAOSTAT_USERNAME, "password": FAOSTAT_PASSWORD},
            timeout=30
        )
        resp.raise_for_status()
        token_data = resp.json()

        if 'AuthenticationResult' in token_data:
            auth_result = token_data['AuthenticationResult']
            FAOSTAT_TOKEN = auth_result.get('IdToken') or auth_result.get('AccessToken')
        else:
            FAOSTAT_TOKEN = token_data.get("access_token") or token_data.get("token")

        FAOSTAT_TOKEN_EXPIRY = time.time() + 3600
        print("  ✅ Token renovado (válido por 60 min)")
        return FAOSTAT_TOKEN
    except Exception as e:
        print(f"  ❌ Error renovando token: {e}")
        raise

# ── LLM (Databricks Foundation Models) ────────────────────────────────────
llm = ChatDatabricks(
    endpoint="databricks-llama-4-maverick",  # Llama 4 Maverick - 400B MoE, 128K context
    temperature=0.0,
    max_tokens=2000,
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
print("✅ Configuración cargada correctamente.")
print(f"   Base de datos    : {DB_NAME}")
print(f"   Tabla de control : {CONTROL_TABLE}")


## Tabla de control de descargas y funciones de hash

In [ ]:
# ── Tabla de control de descargas ─────────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
    fuente          STRING,
    anio            INT,
    nombre_archivo  STRING,
    url_archivo     STRING,
    hash_md5        STRING,
    fecha_descarga  TIMESTAMP
) USING DELTA
""")


def _hash_md5(contenido: bytes) -> str:
    return hashlib.md5(contenido).hexdigest()

def _historial() -> dict:
    """Devuelve dict {nombre_archivo: hash_md5} de todo lo ya descargado."""
    try:
        # Verificar que la tabla existe antes de consultar (evita error Spark Connect)
        if not spark.catalog.tableExists(CONTROL_TABLE):
            return {}
        rows = spark.table(CONTROL_TABLE).select("nombre_archivo","hash_md5").collect()
        return {r["nombre_archivo"]: r["hash_md5"] for r in rows}
    except:
        return {}

def _guardar_y_registrar(fuente, nombre, url, contenido, ruta_vol, anio=0):
    """
    Guarda el archivo si su hash es nuevo.
    Retorna: 'nuevo', 'omitido', o 'error'
    """
    historial   = _historial()
    nuevo_hash  = _hash_md5(contenido)
    if historial.get(nombre) == nuevo_hash:
        print(f"  ⏭ Sin cambios → {nombre}")
        return "omitido"
    ruta_dest = f"{ruta_vol}/{nombre}"
    with open(ruta_dest, "wb") as f:
        f.write(contenido)
    schema = StructType([
        StructField("fuente",         StringType(),  True),
        StructField("anio",           IntegerType(), True),
        StructField("nombre_archivo", StringType(),  True),
        StructField("url_archivo",    StringType(),  True),
        StructField("hash_md5",       StringType(),  True),
        StructField("fecha_descarga", TimestampType(),True),
    ])
    spark.createDataFrame(
        [(fuente, int(anio), nombre, url, nuevo_hash, datetime.now())], schema
    ).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(CONTROL_TABLE)
    print(f"  ✅ Nuevo → {nombre}  ({len(contenido)//1024} KB)")
    return "nuevo"

print("✅ Infraestructura de control de descargas lista.")

## Estado del agente de extracción

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# AGENTE DE EXTRACCIÓN PERSONALIZADO (CUSTOM, SIN FRAMEWORKS DE ORQUESTACIÓN)
# El LLM decide qué archivos son relevantes; el agente decide reintentos y rutas
# de error con lógica de control explícita en Python puro.
# ══════════════════════════════════════════════════════════════════════════

from typing import TypedDict, List, Optional

class ExtractionState(TypedDict):
    """Estado compartido del agente de extracción."""
    # ── Input ─────────────────────────────────────────────────────────────
    fuente_nombre:          str          # ESPAC, SIPA, FAOSTAT, AEBE_BANANOTAS
    fuente_url:             str          # URL base de la fuente
    volumen_destino:        str          # Ruta del volumen donde guardar
    keywords_relevantes:    List[str]    # Palabras clave para filtrar archivos
    
    # ── Paso 1: Listar archivos ──────────────────────────────────────────
    archivos_encontrados:   List[dict]   # [{"nombre": ..., "url": ..., "tamaño": ...}]
    error_listado:          Optional[str]
    
    # ── Paso 2: Consultar LLM para selección ─────────────────────────────
    archivos_seleccionados: List[dict]   # Archivos que el LLM decide descargar
    razonamiento_llm:       str          # Explicación del LLM
    llamadas_api:           int          # Contador M5
    error_seleccion:        Optional[str]
    
    # ── Paso 3: Descargar archivos ───────────────────────────────────────
    archivos_nuevos:        int
    archivos_omitidos:      int
    archivos_error:         int
    kb_descargados:         float
    error_descarga:         Optional[str]
    
    # ── Paso 4: Validar integridad ───────────────────────────────────────
    archivos_validos:       int
    archivos_corruptos:     int
    error_validacion:       Optional[str]
    
    # ── Paso 5: Métricas ─────────────────────────────────────────────────
    timestamp_inicio:       str
    timestamp_fin:          str
    tiempo_segundos:        float
    status:                 str          # OK, ERROR
    error_final:            Optional[str]

print("✅ Estado ExtractionState definido.")

## Pasos 1-3: listar archivos, consultar LLM, descargar

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PASOS DEL AGENTE DE EXTRACCIÓN (1/2)
# ══════════════════════════════════════════════════════════════════════════

def paso_listar_archivos(state: ExtractionState) -> dict:
    """
    PASO 1: Escanea la fuente web y lista archivos disponibles.
    Para cada fuente usa su método específico (scraping, API, etc.)
    """
    print(f"\n{'='*60}")
    print(f"[EXTRACCIÓN PASO 1 — LISTAR] {state['fuente_nombre']}")
    print(f"{'='*60}")
    
    archivos = []
    
    try:
        # ── ESPAC (INEC) ─────────────────────────────────────────────────────
        if state['fuente_nombre'] == 'ESPAC':
            print("  📅 Generando URLs ESPAC Tabulados (2018-actualidad + año siguiente)")
            
            import datetime
            anio_actual = datetime.datetime.now().year
            
            # URLs HARDCODEADAS conocidas (2018-2025) - Patrón verificado
            urls_espac_base = [
                # Años recientes (2023-2025): Patrón /espac/YYYY/
                (2025, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/2025/Tabulados_excel_2025.xlsx"),
                (2024, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/2024/Tabulados_ESPAC_2024.xlsx"),
                (2023, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/2023/Tabulados_ESPAC_2023.xlsx"),
                # Años intermedios (2021-2022): Patrón /espac_YYYY/ o /espac-YYYY/
                (2022, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac_2022/Tabulados ESPAC 2022.xlsx"),
                (2021, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac-2021/Tabulados ESPAC 2021.xlsx"),
                # Años anteriores (2018-2020): Patrón /espac-YYYY/
                (2020, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac-2020/Tabulados ESPAC 2020.xlsx"),
                (2019, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac-2019/Tabulados ESPAC 2019.xlsx"),
                (2018, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac-2018/Tabulados ESPAC 2018.xlsx"),
            ]
            
            # Agregar URLs conocidas
            for anio, url in urls_espac_base:
                nombre = f"ESPAC_Tabulados_excel_{anio}.xlsx"
                archivos.append({"nombre": nombre, "url": url, "tamaño_estimado": "grande", "anio": anio})
                print(f"    ✅ {anio}: URL conocida agregada")
            
            # Intentar predecir URLs para años futuros (año actual + 1)
            # Usar el patrón más reciente: /espac/YYYY/Tabulados_excel_YYYY.xlsx
            anios_futuros = [anio_actual + 1] if anio_actual >= 2025 else []
            
            for anio_futuro in anios_futuros:
                # Intentar 3 patrones posibles
                patrones_posibles = [
                    f"https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/{anio_futuro}/Tabulados_excel_{anio_futuro}.xlsx",
                    f"https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/{anio_futuro}/Tabulados_ESPAC_{anio_futuro}.xlsx",
                    f"https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac-{anio_futuro}/Tabulados ESPAC {anio_futuro}.xlsx",
                ]
                
                for i, url_intento in enumerate(patrones_posibles, 1):
                    try:
                        # Probar HEAD request (rápido)
                        r_test = requests.head(url_intento, headers=HEADERS_HTTP, timeout=10, allow_redirects=True)
                        if r_test.status_code == 200:
                            nombre = f"ESPAC_Tabulados_excel_{anio_futuro}.xlsx"
                            archivos.append({"nombre": nombre, "url": url_intento, "tamaño_estimado": "grande", "anio": anio_futuro})
                            print(f"    ✨ {anio_futuro}: URL futura encontrada (patrón {i})")
                            break  # Encontrado, no probar más patrones
                        else:
                            print(f"    ⚠️  {anio_futuro}: Patrón {i} devuelve {r_test.status_code}")
                    except Exception as e:
                        print(f"    ⚠️  {anio_futuro}: Patrón {i} no accesible ({str(e)[:30]}...)")
                else:
                    print(f"    ℹ️  {anio_futuro}: Ningún patrón funcionó (puede no estar disponible aún)")
            
            print(f"  ✅ Total URLs ESPAC generadas: {len([a for a in archivos if 'ESPAC' in a['nombre']])}")
        
        # ── SIPA (MAG) ──────────────────────────────────────────────────────
        elif state['fuente_nombre'] == 'SIPA':
            # SIPA tiene URLs fijas conocidas
            archivos = [
                {"nombre": "SIPA_TEMPERATURA_PRECIPITACION.xls", 
                 "url": "https://sipa.agricultura.gob.ec/descargas/base-estadistica/modulo_productivo/temperatura-precipitacion.xls",
                 "tamaño_estimado": "medio"},
                {"nombre": "SIPA_USO_SUELO.xlsx",
                 "url": "https://sipa.agricultura.gob.ec/descargas/base-estadistica/modulo_productivo/uso-suelo.xlsx",
                 "tamaño_estimado": "medio"}
            ]
        
        # ── FAOSTAT ────────────────────────────────────────────────────────
        elif state['fuente_nombre'] == 'FAOSTAT':
            # CORRECCIÓN #4: FAOSTAT con fallback inteligente a URLs alternativas
            # CORRECCIÓN #16: Ampliar rango histórico de 7 años a datos desde 1990
            import datetime
            anio_actual = datetime.datetime.now().year
            anio_inicio = 1990  # Datos históricos desde 1990
            
            # Generar lista de años separados por comas
            years = ','.join(str(y) for y in range(anio_inicio, anio_actual + 1))
            
            # Intentar con la URL principal primero
            url_principal = state['fuente_url'] + f"?area=58&item=486,489&year={years}"
            
            # URLs alternativas en caso de que la principal falle
            urls_alternativas = [
                # Servidor principal actual
                url_principal,
                # Servidores alternativos de FAOSTAT
                f"https://fenixservices.fao.org/faostat/api/v1/en/data/QCL?area=58&item=486,489&year={years}",
                f"https://www.fao.org/faostat/api/v1/en/data/QCL?area=58&item=486,489&year={years}",
            ]
            
            # Intentar cada URL hasta que una funcione
            url_exitosa = None
            for i, url_intento in enumerate(urls_alternativas, 1):
                try:
                    print(f"  🔍 FAOSTAT: Probando servidor {i}/{len(urls_alternativas)}...")
                    headers_test = HEADERS_HTTP.copy()
                    token = get_faostat_token()
                    headers_test['Authorization'] = f'Bearer {token}'
                    
                    r_test = requests.head(url_intento, headers=headers_test, timeout=10)
                    if r_test.status_code < 400:
                        url_exitosa = url_intento
                        print(f"  ✅ Servidor {i} accesible")
                        break
                    else:
                        print(f"  ⚠️  Servidor {i} devolvió {r_test.status_code}")
                except Exception as e:
                    print(f"  ❌ Servidor {i} falló: {str(e)[:50]}")
            
            # Si ninguna URL funcionó, usar la principal de todos modos (se manejará en descarga)
            if not url_exitosa:
                print(f"  ⚠️  Ningún servidor respondió, usando URL principal")
                url_exitosa = url_principal
            
            archivos = [{
                "nombre": f"FAOSTAT_BANANAS_PLANTAINS_{anio_inicio}_{anio_actual}.json",
                "url": url_exitosa,
                "tamaño_estimado": "medio"
            }]
            
            print(f"  📅 FAOSTAT: Consulta consolidada ({anio_inicio}-{anio_actual}, 2 productos)")
        
        # ── AEBE (Bananotas) ────────────────────────────────────────────────
        elif state['fuente_nombre'] == 'AEBE_BANANOTAS':
            # Solo se descarga la EDICIÓN MÁS RECIENTE del PDF Bananotas
            # (no todo el histórico), para evitar procesar decenas de PDFs
            # pesados. El sitio lista los PDFs por año en orden cronológico
            # DESCENDENTE, así que el primer PDF único en el HTML completo
            # es la edición más reciente.
            #
            # LIMPIEZA PREVIA: se vacía el volumen raw_aebe antes de descargar,
            # para que el orquestador ETL no reprocese ediciones anteriores
            # junto con la nueva.
            try:
                archivos_previos = [f for f in dbutils.fs.ls(state['volumen_destino'])
                                     if f.name.lower().endswith(".pdf")]
                for f in archivos_previos:
                    dbutils.fs.rm(f.path)
                if archivos_previos:
                    print(f"  🗑️  AEBE: {len(archivos_previos)} PDF(s) de ediciones anteriores eliminados del volumen")
            except Exception as e:
                print(f"  ⚠️  No se pudo limpiar {state['volumen_destino']}: {e}")

            resp = requests.get(state['fuente_url'], headers=HEADERS_HTTP, timeout=30)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "html.parser")

            vistos = set()
            for a in soup.find_all("a", href=True):
                href = a["href"]
                if ".pdf" in href.lower() and "62ee00_" in href:
                    url_completa = href if href.startswith("http") else f"https://www.aebe.com.ec{href}"
                    match = re.search(r'62ee00_([a-f0-9]+)', href)
                    hash_id = match.group(1)[:12] if match else "unknown"
                    if hash_id in vistos:
                        continue  # el mismo PDF se repite varias veces en el HTML (portada + grid)
                    vistos.add(hash_id)
                    nombre = f"AEBE_BANANOTAS_{hash_id}.pdf"
                    archivos.append({"nombre": nombre, "url": url_completa, "tamaño_estimado": "medio"})
                    break  # ⭐ solo la edición más reciente — no seguir recolectando

            print(f"  AEBE Bananotas: {len(archivos)} PDF (solo edición más reciente)")
        
        print(f"  Archivos encontrados: {len(archivos)}")
        for i, arch in enumerate(archivos[:10], 1):  # Mostrar solo los primeros 10
            print(f"    {i}. {arch['nombre'][:60]}")
        if len(archivos) > 10:
            print(f"    ... y {len(archivos)-10} más")
        
        return {"archivos_encontrados": archivos, "error_listado": None}
    
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return {"archivos_encontrados": [], "error_listado": str(e)}


def paso_consultar_llm_seleccion(state: ExtractionState) -> dict:
    """
    PASO 2: Consulta al LLM para decidir qué archivos son relevantes.
    El LLM evalúa nombres de archivos y keywords para priorizar descargas.
    """
    print(f"\n[EXTRACCIÓN PASO 2 — CONSULTAR LLM]")
    
    if not state['archivos_encontrados']:
        print("  ⚠️  No hay archivos para evaluar (listado vacío)")
        return {"archivos_seleccionados": [], "razonamiento_llm": "No hay archivos", 
                "llamadas_api": 0, "error_seleccion": "Sin archivos para evaluar"}
    
    try:
        # Limitar a 100 archivos para no saturar el prompt (aumentado para FAOSTAT dinámico)
        archivos_muestra = state['archivos_encontrados'][:100]
        
        # Preparar lista de archivos como string JSON
        archivos_json = json.dumps(
            [{"nombre": a['nombre'], "url": a['url'][:80]} for a in archivos_muestra], 
            indent=2, 
            ensure_ascii=False
        )
        
        prompt = f"""Eres un experto en datos del sector bananero de Ecuador.

FUENTE: {state['fuente_nombre']}
KEYWORDS RELEVANTES: {', '.join(state['keywords_relevantes'])}

ARCHIVOS ENCONTRADOS ({len(archivos_muestra)} de {len(state['archivos_encontrados'])} totales):
{archivos_json}

TAREA:
1. Identifica qué archivos son relevantes para el sector bananero ecuatoriano
2. Prioriza archivos con datos de producción, precios, superficie, clima
3. Descarta archivos duplicados, de otras provincias irrelevantes, o metadatos

REGLAS ESPECÍFICAS POR FUENTE:
👉 ESPAC (INEC):
   - ⚠️ OBLIGATORIO: Seleccionar TODOS los archivos "Tabulados_excel" o "Tabulados_ESPAC" de TODOS los años (2018-2025+)
   - ⚠️ OBLIGATORIO: Seleccionar TODOS los archivos "Series_historicas" de TODOS los años
   - Estos archivos contienen datos consolidados de banano (T13) y plátano (T26) POR PROVINCIA
   - NO descartar por año - cada año tiene datos únicos necesarios para análisis histórico
   - También incluir: archivos T13 (banano), T26 (plátano), uso del suelo

👉 FAOSTAT:
   - Seleccionar TODOS los años disponibles - datos históricos son críticos
   - NO filtrar por año

👉 AEBE (Bananotas):
   - Solo el PDF de la edición más reciente de la revista Bananotas (rankings de exportadores, marcas, navieras, puertos, mercados)

👉 SIPA:
   - Temperatura, precipitación, uso de suelo

❌ NO descartar archivos con "Tabulados" o "Series" en el nombre
❌ NO filtrar por año - TODOS los años históricos son necesarios

Responde SOLO con JSON válido (sin markdown):
{{
  "archivos_seleccionados": [{{"nombre": "archivo.xlsx", "razon": "motivo"}}],
  "razonamiento": "tu explicación aquí"
}}"""
        
        print(f"  Consultando LLM con {len(archivos_muestra)} archivos...")
        resp = llm.invoke([HumanMessage(content=prompt)])
        llamadas = state.get("llamadas_api", 0) + 1
        
        # Parsear respuesta JSON
        texto = re.sub(r"^```json\s*|^```\s*|```$","",resp.content.strip(),flags=re.MULTILINE).strip()
        start = texto.find("{")
        end = texto.rfind("}")
        if start != -1 and end != -1:
            texto = texto[start:end+1]
            resultado = json.loads(texto)
            
            seleccionados = resultado.get("archivos_seleccionados", [])
            razonamiento = resultado.get("razonamiento", "Sin explicación")
            
            # Mapear nombres a objetos completos
            nombres_sel = {item['nombre'] for item in seleccionados}
            archivos_finales = [a for a in state['archivos_encontrados'] if a['nombre'] in nombres_sel]
            
            print(f"  ✅ LLM seleccionó: {len(archivos_finales)} archivos")
            print(f"  Razonamiento: {razonamiento[:150]}...")
            
            for i, arch in enumerate(archivos_finales[:10], 1):
                print(f"    {i}. {arch['nombre']}")
            if len(archivos_finales) > 10:
                print(f"    ... y {len(archivos_finales)-10} más")
            
            return {"archivos_seleccionados": archivos_finales, 
                    "razonamiento_llm": razonamiento,
                    "llamadas_api": llamadas, 
                    "error_seleccion": None}
    
    except Exception as e:
        print(f"  ⚠️  LLM falló: {e}")
        print(f"  Usando FALLBACK: todos los archivos encontrados")
        return {"archivos_seleccionados": state['archivos_encontrados'],  # SIN LÍMITE
                "razonamiento_llm": f"Fallback: LLM falló, usando todos los archivos ({len(state['archivos_encontrados'])})",
                "llamadas_api": state.get("llamadas_api", 0),
                "error_seleccion": None}  # No es error crítico, el fallback funciona


def paso_descargar_archivos(state: ExtractionState) -> dict:
    """
    PASO 3: Descarga los archivos seleccionados y los guarda en el volumen.
    Verifica duplicados por hash MD5 antes de guardar.
    Para FAOSTAT: descarga secuencial con delays para evitar saturar el servidor.
    """
    print(f"\n[EXTRACCIÓN PASO 3 — DESCARGAR]")
    
    if not state['archivos_seleccionados']:
        print("  ⚠️  No hay archivos seleccionados para descargar")
        return {"archivos_nuevos": 0, "archivos_omitidos": 0, "archivos_error": 0,
                "kb_descargados": 0.0, "error_descarga": "Sin archivos seleccionados"}
    
    nuevos, omitidos, errores = 0, 0, 0
    kb_total = 0.0
    
    try:
        for idx, archivo in enumerate(state['archivos_seleccionados'], 1):
            nombre_original = archivo['nombre']
            url = archivo['url']
            
            try:
                # Prefijo según fuente
                prefijo = {
                    'ESPAC': 'ESPAC_',
                    'SIPA': 'SIPA_' if 'USO' not in nombre_original.upper() else 'ESPAC_',
                    'FAOSTAT': 'FAOSTAT_',
                    'AEBE_BANANOTAS': ''  # el nombre ya viene como AEBE_BANANOTAS_<hash>.pdf
                }.get(state['fuente_nombre'], '')
                
                nombre_limpio = re.sub(r"[^a-zA-Z0-9_\-\.]", "_", nombre_original)
                nombre_final = f"{prefijo}{nombre_limpio}"
                
                # CORRECCIÓN #3: Codificar URL para manejar espacios y caracteres especiales
                from urllib.parse import urlparse, quote
                parsed_url = urlparse(url)
                # Codificar solo el path (no el dominio ni el protocolo)
                path_encoded = quote(parsed_url.path, safe='/:')
                url_encoded = f"{parsed_url.scheme}://{parsed_url.netloc}{path_encoded}"
                if parsed_url.query:
                    url_encoded += f"?{parsed_url.query}"
                
                # CORRECCIÓN #18: FAOSTAT con fallback a servidores alternativos durante descarga
                headers = HEADERS_HTTP.copy()
                contenido = None
                
                if state['fuente_nombre'] == 'FAOSTAT':
                    # Probar la URL seleccionada Y servidores alternativos si falla
                    import datetime
                    anio_actual = datetime.datetime.now().year
                    anio_inicio = 1990
                    years = ','.join(str(y) for y in range(anio_inicio, anio_actual + 1))
                    
                    # URLs alternativas para FAOSTAT
                    urls_alternativas_faostat = [
                        url_encoded,  # La URL seleccionada en el listado
                        f"https://faostatservices.fao.org/api/v1/en/data/QCL?area=58&item=486,489&year={years}",
                        f"https://faostatservices.fao.org/faostat/api/v1/en/data/QCL?area=58&item=486,489&year={years}",
                    ]
                    
                    # Remover duplicados preservando orden
                    urls_unicas = []
                    for u in urls_alternativas_faostat:
                        if u not in urls_unicas:
                            urls_unicas.append(u)
                    
                    # Intentar cada servidor
                    descarga_exitosa = False
                    ultimo_error = None
                    
                    for i, url_intento in enumerate(urls_unicas, 1):
                        try:
                            token = get_faostat_token()
                            headers['Authorization'] = f'Bearer {token}'
                            
                            r = requests.get(url_intento, headers=headers, timeout=90)
                            r.raise_for_status()
                            contenido = r.content
                            descarga_exitosa = True
                            print(f"  🔄 FAOSTAT: Servidor {i} exitoso")
                            break
                        except Exception as e:
                            ultimo_error = e
                            print(f"  ⚠️  FAOSTAT: Servidor {i} falló ({str(e)[:50]}), probando siguiente...")
                    
                    if not descarga_exitosa:
                        raise Exception(f"Todos los servidores FAOSTAT fallaron. Último error: {ultimo_error}")
                    
                else:
                    # Otras fuentes: descarga normal
                    r = requests.get(url_encoded, headers=headers, timeout=90)
                    r.raise_for_status()
                    contenido = r.content
                kb_total += len(contenido) / 1024
                
                # Usar la función existente _guardar_y_registrar
                res = _guardar_y_registrar(
                    state['fuente_nombre'], 
                    nombre_final, 
                    url, 
                    contenido, 
                    state['volumen_destino']
                )
                
                if res == "nuevo":
                    nuevos += 1
                    print(f"  ✅ {nombre_final}")
                elif res == "omitido":
                    omitidos += 1
                    print(f"  ⏭️  {nombre_final} (ya existe)")
            
            except Exception as e:
                errores += 1
                print(f"  ❌ {nombre_original}: {e}")
        
        print(f"\n  Resumen: {nuevos} nuevos, {omitidos} omitidos, {errores} errores ({kb_total:.0f} KB)")
        
        return {"archivos_nuevos": nuevos, 
                "archivos_omitidos": omitidos, 
                "archivos_error": errores,
                "kb_descargados": kb_total, 
                "error_descarga": None}
    
    except Exception as e:
        print(f"  ❌ Error fatal en descarga: {e}")
        return {"archivos_nuevos": nuevos, 
                "archivos_omitidos": omitidos, 
                "archivos_error": errores,
                "kb_descargados": kb_total, 
                "error_descarga": str(e)}

print("✅ Pasos 1-3 del agente de extracción definidos.")

## Pasos 4-5: validar integridad, finalizar y ensamblar el agente

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PASOS DEL AGENTE DE EXTRACCIÓN (2/2) + ENSAMBLADO DEL AGENTE
# ══════════════════════════════════════════════════════════════════════════

def paso_validar_integridad(state: ExtractionState) -> dict:
    """
    PASO 4: Valida que los archivos descargados sean legibles.
    Intenta abrir cada archivo para verificar que no esté corrupto.
    """
    print(f"\n[EXTRACCIÓN PASO 4 — VALIDAR INTEGRIDAD]")
    
    validos, corruptos = 0, 0
    
    try:
        # Listar archivos en el volumen
        archivos_vol = dbutils.fs.ls(state['volumen_destino'])
        print(f"  Validando {len(archivos_vol)} archivos en {state['volumen_destino']}...")
        
        for f in archivos_vol:
            if not f.name.lower().endswith((".xlsx", ".xls", ".csv", ".json")): continue
            
            try:
                path = f.path.replace("dbfs:", "/dbfs")
                
                # Validación básica: verificar que el archivo tiene tamaño > 0
                if f.size > 0:
                    validos += 1
                else:
                    corruptos += 1
                    print(f"  ⚠️  Archivo vacío: {f.name}")
            
            except Exception as e:
                corruptos += 1
                print(f"  ❌ Corrupto: {f.name} - {e}")
        
        print(f"  ✅ Válidos: {validos} | ❌ Corruptos: {corruptos}")
        
        return {"archivos_validos": validos, 
                "archivos_corruptos": corruptos, 
                "error_validacion": None}
    
    except Exception as e:
        print(f"  ❌ Error en validación: {e}")
        return {"archivos_validos": 0, 
                "archivos_corruptos": 0, 
                "error_validacion": str(e)}


def paso_finalizar_extraccion(state: ExtractionState) -> dict:
    """
    PASO 5: Calcula el tiempo total y determina el status final de la
    extracción para esta fuente (sin registrar métricas en Delta).
    """
    ts_inicio = datetime.fromisoformat(state['timestamp_inicio'])
    ts_fin = datetime.now()
    tiempo = (ts_fin - ts_inicio).total_seconds()

    status_final = "OK"
    if state.get('archivos_error', 0) > 0:
        status_final = "PARCIAL"  # Algunos errores pero hubo descargas exitosas
    if state.get('archivos_nuevos', 0) == 0 and state.get('archivos_error', 0) > 0:
        status_final = "ERROR"  # Solo errores, ninguna descarga exitosa

    return {"timestamp_fin": ts_fin.isoformat(),
            "tiempo_segundos": tiempo,
            "status": status_final}


def paso_error_extraccion(state: ExtractionState) -> dict:
    """
    PASO 6: Manejo de errores centralizado para extracción.
    """
    print(f"\n[EXTRACCIÓN PASO ERROR]")
    errores = " | ".join(filter(None, [
        state.get("error_listado"),
        state.get("error_seleccion"),
        state.get("error_descarga"),
        state.get("error_validacion"),
    ]))
    print(f"  Causa: {errores or 'desconocida'}")
    
    return {"status": "ERROR", 
            "error_final": errores or "Error desconocido en extracción"}


# ══════════════════════════════════════════════════════════════════════════
# AGENTE DE EXTRACCIÓN — clase Python pura (sin grafos ni frameworks)
#
# Ejecuta los pasos de extracción en secuencia sobre un estado compartido
# (un diccionario). Si un paso falla, el agente corta la cadena y delega en
# paso_error_extraccion — el mismo comportamiento que antes ofrecían las
# aristas condicionales, expresado aquí como simples condicionales en Python.
# ══════════════════════════════════════════════════════════════════════════

class AgenteExtraccion:
    """Agente personalizado que orquesta la extracción de una fuente de datos."""

    def _hubo_error_tras_listar(self, estado: dict) -> bool:
        return bool(estado.get("error_listado"))

    def _hubo_error_tras_seleccion(self, estado: dict) -> bool:
        return bool(estado.get("error_seleccion") and not estado.get("archivos_seleccionados"))

    def _hubo_error_tras_descarga(self, estado: dict) -> bool:
        return bool(estado.get("error_descarga") and estado.get("archivos_nuevos", 0) == 0)

    def _hubo_error_tras_validacion(self, estado: dict) -> bool:
        return bool(estado.get("error_validacion"))

    def ejecutar(self, estado_inicial: dict) -> dict:
        """Corre el flujo completo: listar → seleccionar (LLM) → descargar → validar → métricas."""
        estado = dict(estado_inicial)

        estado.update(paso_listar_archivos(estado))
        if self._hubo_error_tras_listar(estado):
            estado.update(paso_error_extraccion(estado))
            return estado

        estado.update(paso_consultar_llm_seleccion(estado))
        if self._hubo_error_tras_seleccion(estado):
            estado.update(paso_error_extraccion(estado))
            return estado

        estado.update(paso_descargar_archivos(estado))
        if self._hubo_error_tras_descarga(estado):
            estado.update(paso_error_extraccion(estado))
            return estado

        estado.update(paso_validar_integridad(estado))
        if self._hubo_error_tras_validacion(estado):
            estado.update(paso_error_extraccion(estado))
            return estado

        estado.update(paso_finalizar_extraccion(estado))
        return estado


agente_extraccion = AgenteExtraccion()

print("✅ Pasos 4-5 del agente de extracción definidos.")
print("✅ Agente de extracción ensamblado (clase AgenteExtraccion, sin frameworks de grafos).")
print("   Flujo: listar → consultar_llm → descargar → validar → finalizar (con manejo de error en cada paso)")


## Orquestador: ejecuta la extracción para las 4 fuentes

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ORQUESTADOR DEL AGENTE DE EXTRACCIÓN
# Ejecuta el agente personalizado para cada una de las 4 fuentes de datos.
# ══════════════════════════════════════════════════════════════════════════

print("="*70)
print("📥 AGENTE DE EXTRACCIÓN PERSONALIZADO — INICIANDO")
print("="*70)

# Configuración de las 4 fuentes
FUENTES_CONFIG = [
    {
        "nombre": "ESPAC",
        "url": "https://www.ecuadorencifras.gob.ec/estadisticas-agropecuarias-2/",
        "volumen": RAW_PATH_ESPAC,
        "keywords": ["banano", "platano", "T13", "T26", "uso", "suelo", "cultivos", "permanentes"]
    },
    {
        "nombre": "SIPA",
        "url": "https://sipa.agricultura.gob.ec/",
        "volumen": RAW_PATH_SIPA,
        "keywords": ["temperatura", "precipitacion", "clima", "uso", "suelo"]
    },
    {
        "nombre": "FAOSTAT",
        "url": "https://faostatservices.fao.org/api/v1/en/data/QCL",
        "volumen": RAW_PATH_FAOSTAT,
        "keywords": ["bananas", "plantains", "production", "ecuador"]
    },
    {
        "nombre": "AEBE_BANANOTAS",
        "url": "https://www.aebe.com.ec/bananotas",
        "volumen": RAW_PATH_AEBE,
        "keywords": ["exportaciones", "regiones", "estadisticas", "cajas", "participacion"]
    },
]

resultados_extraccion = []

for fuente_cfg in FUENTES_CONFIG:
    print(f"\n{'#'*70}")
    print(f"📥 FUENTE: {fuente_cfg['nombre']}")
    print(f"{'#'*70}")
    
    # Estado inicial para esta fuente
    estado_inicial: ExtractionState = {
        "fuente_nombre":        fuente_cfg["nombre"],
        "fuente_url":           fuente_cfg["url"],
        "volumen_destino":      fuente_cfg["volumen"],
        "keywords_relevantes":  fuente_cfg["keywords"],
        
        "archivos_encontrados": [],
        "error_listado":        None,
        
        "archivos_seleccionados": [],
        "razonamiento_llm":     "",
        "llamadas_api":         0,
        "error_seleccion":      None,
        
        "archivos_nuevos":      0,
        "archivos_omitidos":    0,
        "archivos_error":       0,
        "kb_descargados":       0.0,
        "error_descarga":       None,
        
        "archivos_validos":     0,
        "archivos_corruptos":   0,
        "error_validacion":     None,
        
        "timestamp_inicio":     datetime.now().isoformat(),
        "timestamp_fin":        "",
        "tiempo_segundos":      0.0,
        "status":               "PENDIENTE",
        "error_final":          None,
    }
    
    try:
        # Ejecutar el agente de extracción
        estado_final = agente_extraccion.ejecutar(estado_inicial)
        resultados_extraccion.append(estado_final)
        
        icono = "✅" if estado_final["status"] == "OK" else "❌"
        print(f"\n{icono} {fuente_cfg['nombre']} → {estado_final['status']}")
        print(f"   Nuevos: {estado_final.get('archivos_nuevos',0)} | "
              f"Omitidos: {estado_final.get('archivos_omitidos',0)} | "
              f"Errores: {estado_final.get('archivos_error',0)}")
        print(f"   Tiempo: {estado_final.get('tiempo_segundos',0):.1f}s | "
              f"KB descargados: {estado_final.get('kb_descargados',0):.0f}")
        if estado_final.get('llamadas_api', 0) > 0:
            print(f"   Llamadas LLM: {estado_final.get('llamadas_api',0)}")
            print(f"   Razonamiento: {estado_final.get('razonamiento_llm','')[:100]}...")
    
    except Exception as e:
        print(f"❌ Error fatal en {fuente_cfg['nombre']}: {e}")
        resultados_extraccion.append({"fuente_nombre": fuente_cfg["nombre"], "status": "ERROR", "error_final": str(e)})

exitosos = sum(1 for r in resultados_extraccion if r.get("status") == "OK")
total_nuevos = sum(r.get("archivos_nuevos", 0) for r in resultados_extraccion)
total_llamadas_llm = sum(r.get("llamadas_api", 0) for r in resultados_extraccion)

print(f"\n{'='*70}")
print(f"🎉 AGENTE DE EXTRACCIÓN FINALIZADO")
print(f"{'='*70}")
print(f"  Fuentes exitosas: {exitosos}/{len(FUENTES_CONFIG)}")
print(f"  Total archivos nuevos: {total_nuevos}")
print(f"  Total llamadas LLM: {total_llamadas_llm}")
print(f"{'='*70}")

# ========== EXPORTAR TASK VALUES (OBLIGATORIO PARA EL JOB) ==========
hay_nuevos = "true" if total_nuevos > 0 else "false"
dbutils.jobs.taskValues.set(key="hay_nuevos", value=hay_nuevos)
dbutils.jobs.taskValues.set(key="total_archivos", value=total_nuevos)
print(f"✅ Task Values exportados: hay_nuevos={hay_nuevos}, total_archivos={total_nuevos}")